<a href="https://colab.research.google.com/github/batinylmz/universiteSikayetSistemi/blob/main/oneClassSVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---------------- FİYAT TRENDLERİNDEN ARINDIRILMIŞ (STATIONARY) OCSVM ----------------

%pip install pyod -q

import pandas as pd
import numpy as np
import time
from tqdm import tqdm
from sklearn.preprocessing import RobustScaler
from pyod.models.ocsvm import OCSVM
from sklearn.metrics import precision_recall_curve, auc, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("1. Veri yükleniyor ve temizleniyor...")
file_name = "/content/drive/MyDrive/financial_anomaly_benchmark_data (1).csv" # Kendi yolun
df = pd.read_csv(file_name)

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.set_index('Timestamp', inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.ffill().fillna(0)

# Gerçek Anomali Etiketleri (%1)
df['Shock_Score'] = df['Volatility_HighLow'] * df['Volume_Change'].abs()
threshold = df['Shock_Score'].quantile(0.99)
df['y_true'] = (df['Shock_Score'] >= threshold).astype(int)

# 💡 İŞTE SİHİRLİ DOKUNUŞ BURASI 💡
# Ham fiyatları (Open, High vb.) attık! Sadece anlık şokları gösteren durağan özellikleri bıraktık.
features = ['Returns', 'Volume_Change', 'Volatility_HighLow']
X = df[features]
y = df['y_true']

# KAYAN PENCERE AYARLARI
train_window_size = 2880  # Geçmiş 30 gün
test_window_size = 96     # Gelecek 1 gün
test_indices = np.arange(train_window_size, len(df), test_window_size)

y_test_real = []
y_test_scores = []
y_test_preds = []

print(f"2. Durağan (Stationary) OCSVM Başlatılıyor ({len(test_indices)} döngü)...")
start_time = time.time()

for start_test_idx in tqdm(test_indices, desc="Durağan OCSVM Eğitimi"):

    start_train_idx = start_test_idx - train_window_size
    end_test_idx = min(start_test_idx + test_window_size, len(df))

    X_train_window = X.iloc[start_train_idx:start_test_idx]
    X_test_window = X.iloc[start_test_idx:end_test_idx]
    y_test_window = y.iloc[start_test_idx:end_test_idx]

    # 1. Aşama: OCSVM için Robust Ölçeklendirme
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_window)
    X_test_scaled = scaler.transform(X_test_window)

    # 2. Aşama: OCSVM Eğitimi (Veri artık saf bir top olduğu için nu=0.01'e geri dönebiliriz)
    model_ocsvm = OCSVM(kernel='rbf', nu=0.01, gamma='scale')
    model_ocsvm.fit(X_train_scaled)

    # 3. Aşama: Tahminler ve Olasılıklar
    train_scores = model_ocsvm.decision_scores_
    test_scores = model_ocsvm.decision_function(X_test_scaled)

    # 4. Aşama: Dinamik Eşik (%99)
    dynamic_threshold = np.percentile(train_scores, 99)
    test_preds = (test_scores >= dynamic_threshold).astype(int)

    y_test_real.extend(y_test_window.values)
    y_test_scores.extend(test_scores)
    y_test_preds.extend(test_preds)

end_time = time.time()
inference_time_ms = ((end_time - start_time) / len(test_indices)) * 1000

print("\n3. Başarı Metrikleri Hesaplanıyor...")
roc_auc = roc_auc_score(y_test_real, y_test_scores)
precision, recall, _ = precision_recall_curve(y_test_real, y_test_scores)
final_pr_auc = auc(recall, precision)
final_f1 = f1_score(y_test_real, y_test_preds)

print("\n" + "═"*55)
print(" 🚀 DURAĞAN FİNANSAL OCSVM SONUÇLARI 🚀")
print("═"*55)
print(f"ROC-AUC Skoru                : {roc_auc:.4f}")
print(f"PR-AUC Skoru                 : {final_pr_auc:.4f} (Şimdi uçuşa geçmeli!)")
print(f"F1 Skoru                     : {final_f1:.4f} (Yüksek kalacaktır)")
print(f"Ort. Çıkarım Süresi (Pencere): {inference_time_ms:.2f} milisaniye")
print("═"*55)

# CSV Güncelleme
results_df = pd.DataFrame({
    'Model': ['Stationary Sliding Window OCSVM'],
    'ROC_AUC': [roc_auc],
    'PR_AUC': [final_pr_auc],
    'F1_Score': [final_f1],
    'Inference_Time_ms': [inference_time_ms]
})
results_df.to_csv('/content/drive/MyDrive/benchmark_results2.csv', mode='a', header=False, index=False)
print("✅ En iyi OCSVM sonuçları CSV'ye kaydedildi!")

1. Veri yükleniyor ve temizleniyor...
2. Durağan (Stationary) OCSVM Başlatılıyor (1439 döngü)...


Durağan OCSVM Eğitimi: 100%|██████████| 1439/1439 [00:37<00:00, 38.55it/s]



3. Başarı Metrikleri Hesaplanıyor...

═══════════════════════════════════════════════════════
 🚀 DURAĞAN FİNANSAL OCSVM SONUÇLARI 🚀
═══════════════════════════════════════════════════════
ROC-AUC Skoru                : 0.9555
PR-AUC Skoru                 : 0.5081 (Şimdi uçuşa geçmeli!)
F1 Skoru                     : 0.4005 (Yüksek kalacaktır)
Ort. Çıkarım Süresi (Pencere): 25.94 milisaniye
═══════════════════════════════════════════════════════
✅ En iyi OCSVM sonuçları CSV'ye kaydedildi!
